<a href="https://colab.research.google.com/github/Ayon150/AI_-_DL/blob/main/AI_%26_DL_ASSIGNMENT_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# TASK-1 : YOLOv8 TRAINING USING LOCAL ANNOTATED DATASET
# GOOGLE COLAB READY CODE
# ============================================================

# ============================================================
# STEP 1 : CHECK GPU
# ============================================================

!nvidia-smi


# ============================================================
# STEP 2 : INSTALL YOLOv8
# ============================================================

!pip install -q ultralytics


# ============================================================
# STEP 3 : IMPORT LIBRARIES
# ============================================================

from ultralytics import YOLO
from google.colab import files

import os
import zipfile
import shutil
import cv2
import matplotlib.pyplot as plt


# ============================================================
# STEP 4 : UPLOAD DATASET ZIP
# ============================================================
# Upload the ZIP file you downloaded from Roboflow

uploaded = files.upload()


# ============================================================
# STEP 5 : EXTRACT DATASET
# ============================================================

zip_file = list(uploaded.keys())[0]

extract_path = "/content/dataset"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset Extracted Successfully")


# ============================================================
# STEP 6 : CHECK DATASET STRUCTURE
# ============================================================

for root, dirs, files_in_dir in os.walk(extract_path):

    print(root)

    if len(files_in_dir) > 0:
        print(files_in_dir[:3])

    print("------------------------------------------------")


# ============================================================
# STEP 7 : FIND data.yaml FILE
# ============================================================

yaml_path = None

for root, dirs, files_in_dir in os.walk(extract_path):

    for file in files_in_dir:

        if file == "data.yaml":

            yaml_path = os.path.join(root, file)

            break

print("\nDATA.YAML PATH:")
print(yaml_path)


# ============================================================
# STEP 8 : SHOW SAMPLE TRAIN IMAGE
# ============================================================

train_images_path = None

for root, dirs, files_in_dir in os.walk(extract_path):

    if "train" in root and "images" in root:

        train_images_path = root

        break

print("\nTRAIN IMAGE PATH:")
print(train_images_path)

sample_image = os.listdir(train_images_path)[0]

sample_image_path = os.path.join(train_images_path, sample_image)

img = cv2.imread(sample_image_path)

img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10,10))
plt.imshow(img)
plt.title("Sample Training Image")
plt.axis("off")
plt.show()


# ============================================================
# STEP 9 : LOAD PRETRAINED YOLOv8 MODEL
# ============================================================

model = YOLO("yolov8n.pt")


# ============================================================
# STEP 10 : TRAIN MODEL
# ============================================================

results = model.train(

    data=yaml_path,

    epochs=50,

    imgsz=640,

    batch=16,

    device=0,

    workers=2,

    project="Task1_Object_Detection",

    name="YOLOv8_Custom",

    save=True,

    # ============================================
    # DATA AUGMENTATION
    # ============================================

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=2,

    fliplr=0.5,

    mosaic=1.0,

    mixup=0.2,

    patience=20
)


# ============================================================
# STEP 11 : VALIDATION
# ============================================================

metrics = model.val()

print("\n================================")
print("VALIDATION RESULTS")
print("================================")

print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")
print(f"mAP50     : {metrics.box.map50:.4f}")
print(f"mAP50-95  : {metrics.box.map:.4f}")


# ============================================================
# STEP 12 : SHOW TRAINING RESULTS
# ============================================================

results_image_path = "/content/Task1_Object_Detection/YOLOv8_Custom/results.png"

if os.path.exists(results_image_path):

    img = cv2.imread(results_image_path)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(15,10))
    plt.imshow(img)
    plt.title("Training Results")
    plt.axis("off")
    plt.show()

else:

    print("results.png not found")


# ============================================================
# STEP 13 : SHOW CONFUSION MATRIX
# ============================================================

confusion_path = "/content/Task1_Object_Detection/YOLOv8_Custom/confusion_matrix.png"

if os.path.exists(confusion_path):

    img = cv2.imread(confusion_path)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10,10))
    plt.imshow(img)
    plt.title("Confusion Matrix")
    plt.axis("off")
    plt.show()

else:

    print("confusion_matrix.png not found")


# ============================================================
# STEP 14 : UPLOAD TEST IMAGE
# ============================================================

print("\nUpload a custom image for testing")

uploaded_test = files.upload()

test_image = list(uploaded_test.keys())[0]

print("Test Image:", test_image)


# ============================================================
# STEP 15 : LOAD BEST MODEL
# ============================================================

best_model_path = "/content/Task1_Object_Detection/YOLOv8_Custom/weights/best.pt"

trained_model = YOLO(best_model_path)


# ============================================================
# STEP 16 : RUN DETECTION
# ============================================================

results = trained_model.predict(

    source=test_image,

    conf=0.25,

    save=True,

    show=True
)


# ============================================================
# STEP 17 : DISPLAY DETECTION RESULT
# ============================================================

save_dir = results[0].save_dir

predicted_images = os.listdir(save_dir)

predicted_image_path = os.path.join(save_dir, predicted_images[0])

img = cv2.imread(predicted_image_path)

img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12,12))
plt.imshow(img)
plt.title("Detection Result")
plt.axis("off")
plt.show()


# ============================================================
# STEP 18 : TEST ON VIDEO
# ============================================================

print("\nUpload video for object detection")

uploaded_video = files.upload()

video_file = list(uploaded_video.keys())[0]

video_results = trained_model.predict(

    source=video_file,

    conf=0.25,

    save=True
)

print("Video Detection Completed")


# ============================================================
# STEP 19 : TEST SET EVALUATION
# ============================================================

test_metrics = trained_model.val(

    data=yaml_path,

    split="test"
)

print("\n================================")
print("TEST RESULTS")
print("================================")

print(f"Precision : {test_metrics.box.mp:.4f}")
print(f"Recall    : {test_metrics.box.mr:.4f}")
print(f"mAP50     : {test_metrics.box.map50:.4f}")
print(f"mAP50-95  : {test_metrics.box.map:.4f}")


# ============================================================
# STEP 20 : INFERENCE SPEED TEST
# ============================================================

print("\n================================")
print("INFERENCE SPEED")
print("================================")

speed_results = trained_model.predict(

    source=test_image,

    conf=0.25,

    verbose=True
)


# ============================================================
# STEP 21 : EXPORT MODEL
# ============================================================

trained_model.export(format="onnx")

print("\nONNX MODEL EXPORTED")


# ============================================================
# STEP 22 : DOWNLOAD BEST MODEL
# ============================================================

files.download(best_model_path)


# ============================================================
# STEP 23 : FINAL SUMMARY
# ============================================================

print("\n================================================")
print("TASK-1 COMPLETED SUCCESSFULLY")
print("================================================")

print("Custom Dataset Loaded")
print("YOLOv8 Fine-Tuning Completed")
print("Validation Completed")
print("Testing Completed")
print("Inference Speed Measured")
print("Image Detection Completed")
print("Video Detection Completed")
print("Model Exported")
print("Ready For Report Writing")

Mon May 18 05:03:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----